# MICrONS Option 3 — Time-resolved PCA trajectories

Per-area, per-stimulus trial-averaged trajectories with the time axis preserved. Resolves the Monet2/Trippy collapse seen in Option 2 by keeping within-trial dynamics. See `docs/specs/2026-05-04-option3-trajectories-design.md`.

**Sections will be filled in by subsequent tasks.**

In [ ]:
# === Smoke test: sections (a), (b), (c) on real session 7_5 data ===
# This cell is REPLACED by proper Part 0 in Task 3.
import os
from pathlib import Path

import numpy as np
import microns_eda
import option2_pca_utils
import option3_trajectories_utils as traj_utils

DATADIR = Path(os.environ.get("MICRONS_DATADIR", "../neuroscience"))
SESSION = "7_5"
RANDOM_SEED = 42

reader = microns_eda.open_dataset(DATADIR)
responses, trial_boundaries, _ = microns_eda.load_session_responses(
    reader, DATADIR, SESSION
)
meta = microns_eda.get_session_meta(DATADIR, SESSION)

# Behavioral cleanup (reuse Option 2's path).
n_trials = meta["n_trials"]
per_trial_tread_means = np.empty(n_trials, dtype=np.float64)
for i in range(n_trials):
    trial = microns_eda.load_trial(reader, DATADIR, SESSION, i)
    per_trial_tread_means[i] = np.nanmean(trial["treadmill"])
clean_trial_indices, _ = microns_eda.compute_clean_trial_indices(
    per_trial_tread_means, threshold=1.0
)

# Stim labels for the clean subset.
stim_map = microns_eda.build_stim_type_map(reader, SESSION)
stim_types_full = np.array([
    stim_map[h.decode() if isinstance(h, bytes) else h]
    for h in meta["condition_hashes"]
])
labels = stim_types_full[clean_trial_indices]

# Apply Stage A preprocessing (reuses Option 2).
responses_pp = option2_pca_utils.preprocess_responses(responses, apply_log=False)

# Section (a): build trajectories.
trajectories = traj_utils.build_stim_trajectories(
    responses_pp, trial_boundaries, clean_trial_indices,
    meta["brain_areas"], labels, n_frames=75,
)
print("=== build_stim_trajectories ===")
for area in sorted(trajectories.keys()):
    for stim in sorted(trajectories[area].keys()):
        shape = trajectories[area][stim].shape
        print(f"  {area} / {stim}: shape {shape}")
assert set(trajectories.keys()) == {"V1", "AL", "LM", "RL"}
for area in trajectories:
    assert set(trajectories[area].keys()) == {"Clip", "Monet2", "Trippy"}

# Section (b): fit PCA per area.
pca_per_area = traj_utils.fit_trajectory_pca(trajectories, n_components=10)
print("\n=== fit_trajectory_pca ===")
for area in sorted(pca_per_area.keys()):
    pca = pca_per_area[area]["pca"]
    print(f"  {area} top-3 cum var = {100 * pca.explained_variance_ratio_[:3].sum():.1f}%")

# Section (c): pairwise distance for V1.
v1_distances = traj_utils.pairwise_trajectory_distance(
    pca_per_area["V1"]["stim_pcs"], metric="full"  # placeholder — uses PC-projected trajs
)
# Actual full-feature distance on raw V1 trajectories.
v1_distances_full = traj_utils.pairwise_trajectory_distance(
    trajectories["V1"], metric="full"
)
print("\n=== V1 full-feature pairwise distances (max across frames) ===")
for pair, d in v1_distances_full.items():
    print(f"  {sorted(pair)}: max distance = {d.max():.3f}")
print("smoke test OK")